# Stream candidate investigation: example

One stream candidate in Euclid DR1 south (Euclid only, no DES), from two points along it: (ra, dec) = (77, −40) and (70, −43) deg.

The investigation is held by one object, `CandidateInvestigation`: it keeps every input and result and calls the functions in order. Each cell below runs one step, so its result can be looked at and tuned before the next one. Where a step needs more tuning than the arguments of the class, the cell calls the function directly and puts the result back in the object (e.g. `cand.regions = stream_regions(...)`). The last section does everything at once with `cand.run_all_steps(...)`.

Steps:
1. inputs;
2. frame and window;
3. catalog and matched-filter map of the loaded box;
4. frame check: move the endpoints by hand until the frame follows the stream;
5. matched-filter maps over distance moduli, in the frame;
6. width;
7. on and off regions;
8. Hess diagrams (CMDs) on, off and on − off, in stars/deg²;
9. isochrone;
10. summary;
11. all at once.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from streameucliddes.analysis.candidate_investigation import CandidateInvestigation
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.plotting.plot_generic_search import plot_generic_search_residual_map
from streameucliddes.stream_frames import phi12_to_radec, phi12_window_outline, radec_to_phi12

## 1. Inputs

All the inputs in one cell: the name, the two endpoints, the isochrone (13 Gyr, Z = 0.0002 by default: `default_isochrone_grid()[0]`), the match filter (VIS vs VIS − H by default) and the sizes of the regions.

In [ ]:
NAME = "euclid_dr1_south_candidate"
ENDPOINT_1 = (77.0, -40.5)  # (ra, dec) deg
ENDPOINT_2 = (70.0, -44.5)

CATALOG = "euclid_dr1"  # Euclid DR1 stars, Euclid bands only
STAR_SELECTION = "morphology_cuts"  # or "xgboost_classifier" (see default_star_cuts)
LOAD_MARGIN_DEG = 5.0  # stars read in the (ra, dec) box of the window plus this margin: room to move the endpoints
SKY_MAP_DM = 17.25  # distance modulus of the map of the loaded box: where the candidate shows up
MATCH_FILTER = {  # Euclid VIS vs VIS - H, with the DR1 columns
    "name": "euclid_dr1_vis_h",
    "reference_band": ("euclid_dr1", "VIS"),
    "color_bands": [("euclid_dr1", "H")],
}

cand = CandidateInvestigation(
    NAME, ENDPOINT_1, ENDPOINT_2,
    age=13.0, metallicity=0.0002,  # Gyr, Z; or feh=... instead of metallicity
    match_filter=MATCH_FILTER,
)
cand.isochrone

## 2. Frame and window

Set at construction: the great-circle frame through the endpoints (`cand.stream_frame`, gala `GreatCircleICRSFrame.from_endpoints`: phi1 = 0 at their midpoint), the phi1/phi2 window around them (`cand.window`) and the (ra, dec) box of the window (`cand.radec_box`).

In [ ]:
phi1, phi2 = radec_to_phi12(
    [cand.endpoint_1[0], cand.endpoint_2[0]], [cand.endpoint_1[1], cand.endpoint_2[1]], cand.stream_frame
)
print(f"Endpoints in the frame: phi1 = {np.round(phi1, 2)} deg, |phi2| <= {np.abs(phi2).max():.0e} deg")
print(f"Window (phi1_min, phi1_max, phi2_min, phi2_max): {np.round(cand.window, 2)} deg")
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
print(f"(ra, dec) box of the window: ra {ra_min:.1f} to {ra_max:.1f} deg, dec {dec_min:.1f} to {dec_max:.1f} deg")

## 3. Catalog and matched-filter map of the loaded box

Euclid DR1 south, read once in a box generously larger than the candidate, so that the endpoints can move without reading again. The "sky map" is the matched-filter map of this box (`generic_search_single_dm`), with the great circle of the frame. It does not depend on the frame: computed once.

In [ ]:
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
ra_range = ((ra_min - LOAD_MARGIN_DEG) % 360.0, (ra_max + LOAD_MARGIN_DEG) % 360.0)  # ra_min > ra_max: across ra = 0
dec_range = (max(dec_min - LOAD_MARGIN_DEG, -90.0), min(dec_max + LOAD_MARGIN_DEG, 90.0))
print(f"Loaded box: ra {ra_range[0]:.1f} to {ra_range[1]:.1f} deg, dec {dec_range[0]:.1f} to {dec_range[1]:.1f} deg")

catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), ra_range=ra_range, dec_range=dec_range)

In [ ]:
cand.compute_sky_map(catalog, SKY_MAP_DM)
print(f"{cand.sky_map['n_selected']:,} stars in the match filter at dm {SKY_MAP_DM}")

phi1_circle = np.linspace(-180.0, 180.0, 721)
great_circle = phi12_to_radec(phi1_circle, np.zeros_like(phi1_circle), cand.stream_frame)
window_outline = phi12_to_radec(*phi12_window_outline(cand.window), cand.stream_frame)
fig, ax = plot_generic_search_residual_map(
    cand.sky_map, tracks=[("great circle", *great_circle), ("window", *window_outline)]
)

## 4. Frame check

The stars of the window are prepared (`cand.prepare()`), and one map is made in the frame at the distance modulus of the sky map (`cand.compute_zoom_map()`). `plot_candidate_frame_check` shows, side by side, the sky map with the great circle, the window and the endpoints, and the zoomed map in phi1/phi2.

Edit the endpoints in the next cell and run it again, until the stream lies along phi2 ≈ 0. The window must contain the off-stream bands: checked with an `assert`.

## 5. Matched-filter maps over distance moduli, in the frame

`cand.scan_distance_moduli()`, then a few maps with `plot_candidate_map`. Keep the best distance modulus by eye: `cand.distance_modulus = ...`.

## 6. Width

Stars in the match filter at `cand.distance_modulus` → phi2 density profile (stars/deg², masked sky left out) → Gaussian plus constant background (`fit_stream_width`, Shipp et al. 2018) → `plot_phi2_profile`.

## 7. Regions

On region: |phi2 − phi2_centre| ≤ 1.4σ; off region: two bands at 2.5σ to 4.5σ, both over `phi1_range`. `phi1_range` first by eye, then from the phi1 profile of on − off densities (`on_off_phi1_profile`). The regions on the map (`draw_stream_regions`) and their areas outside the masks (`region_area_deg2`).

## 8. Hess diagrams

On and off Hess diagrams with the same bins (`hess_diagram`), in stars/deg²/bin, and their difference (`difference_hess`): `plot_on_off_difference_hess`, then the difference alone with `plot_hess(difference=True)`.

## 9. Isochrone

The isochrone in the bands of the match filter (`isochrone_cmd_track`, the same ugali model as the match filter) over the difference Hess diagram (`plot_isochrone`), with the match-filter polygon if wanted (`plot_match_filter_polygon`). Try 2–3 (age, [Fe/H], distance modulus).

## 10. Summary

`cand.summary()` and `plot_candidate_overview(cand)`.

## 11. All at once

A new `CandidateInvestigation` with the chosen inputs, and `cand.run_all_steps(catalog)`: it should give the same results as above.